# Python 绘图排版指南：尺寸、分辨率、字号与 Quarto 适配

> 本章面向使用 Jupyter Notebook 写作、使用 Quarto 编译 OnlineBook、并可能导出 A4 PDF 的读者。目标不是穷尽 matplotlib 的所有参数，而是给出一套可复用的绘图排版工作流。

本章配套文件：

- `data_visual_setting_lec.ipynb`：讲义正文。
- `data_visual_setting_codes.ipynb`：配套代码，用于生成本章所有配图。
- `./figs/`：图片输出文件夹。
- `./data/`：示例数据输出文件夹。

建议先运行 `data_visual_setting_codes.ipynb`，生成所有图片，再渲染本讲义。

## 快速导航

| 遇到的问题 | 建议阅读 |
|---|---|
| Notebook 里正常，网页或 PDF 中变样 | 第 1 节 |
| 不清楚 `figsize`、`dpi`、`width` 的区别 | 第 2 节 |
| 字号设成 11，但图例看起来忽大忽小 | 第 2.4 节、第 3.2 节 |
| 单图、双图、三图不知道如何设置 | 第 4 节 |
| 多章节 OnlineBook 图片风格不统一 | 第 5 节 |
| 希望直接复制一套配置代码 | 第 6 节 |
| 希望用 AI 生成适合自己的配置 | 第 7 节 |
| PNG、SVG、PDF 不知道怎么选 | 第 8 节 |

## 1. 问题从哪里来？

很多 Python 初学者会遇到类似问题：Notebook 里看到的图很清楚，放到 Quarto 网页中却显得过大；导出 A4 PDF 后，三张子图被压得很小；手机端打开时，图例和坐标轴几乎看不清。

这类问题通常不是单个参数设置错误，而是因为同一张图会经过多个环节：

![](./figs/data_visual_setting_fig01_workflow.png){width="92%"}

### 1.1 Python 负责生成图片，Quarto 负责显示图片

在 Python 中，`figsize`、`dpi`、`fontsize` 等参数控制图片如何生成。  
在 Quarto 中，`width="85%"`、`fig-cap` 等设置控制图片如何显示。  
在浏览器、PDF 阅读器和手机屏幕中，图片还会根据设备宽度进一步缩放。

因此，同一张图的最终效果不是由 Python 单独决定的，而是由以下三层共同决定：

- **生成层**：Python 生成多大的图、多高的分辨率、图中文字多大。
- **排版层**：Quarto 把图片放到页面中多宽。
- **显示层**：浏览器、PDF 阅读器和屏幕设备如何缩放图片。

### 1.2 初学者最常见的三个误区

- **误区一：以为 `dpi` 越高，图片一定越好。**  
  `dpi` 可以提高 PNG 的像素密度，但如果图中文字太小，单纯提高 `dpi` 并不能改善可读性。

- **误区二：以为 `fontsize=11` 就等于正文 11 pt。**  
  图中的 `fontsize=11` 会随着图片整体缩放而缩放。最终视觉大小还取决于图片在文档中的显示宽度。

- **误区三：以为 Notebook 里好看，Quarto 编译后就一定好看。**  
  Notebook 的内联显示有自己的缩放规则，不能完全代表网页和 PDF 中的最终效果。

本章的核心思路是：先理解图片如何生成，再理解图片如何显示，最后用标准配置把风格固定下来。

## 2. 三个核心概念：尺寸、分辨率、显示宽度

### 2.1 `figsize`：图片生成时的物理尺寸

在 matplotlib 中，常见写法是：

```python
fig, ax = plt.subplots(figsize=(7.0, 4.2))
```

这里的 `7.0` 和 `4.2` 单位是 inch，不是 cm，也不是 pixel。`figsize` 决定图形画布的宽高比例，也会影响图内元素的拥挤程度。

例如，同样的坐标轴标签、图例和标题，如果画布太窄，就容易挤在一起。

### 2.2 `dpi`：图片导出时的像素密度

保存 PNG 图片时，常见写法是：

```python
fig.savefig("./figs/demo.png", dpi=220)
```

PNG 的像素宽度大致为：

$$
\text{pixel width}
=
\text{figsize width}
\times
\text{dpi}
$$

例如，当 `figsize=(7.0, 4.2)`，且 `dpi=220` 时，图片宽度大约为：

$$
7.0 \times 220 = 1540 \text{ px}
$$

这对网页和高分屏设备通常已经比较稳。

### 2.3 `width`：Quarto 中的最终显示宽度

在 Quarto 或 Markdown 中，插入图片时可以写：

```markdown
![](./figs/demo.png){width="85%"}
```

这里的 `width="85%"` 控制的是图片在文档中的显示宽度，而不是重新生成图片。

### 2.4 三者共同决定最终效果

![](./figs/data_visual_setting_fig02_figsize_dpi_width.png){width="88%"}

可以把三者的关系理解为：

| 参数 | 所在位置 | 控制对象 | 主要影响 |
|---|---|---|---|
| `figsize` | Python | 图形画布尺寸 | 图形比例、元素拥挤程度 |
| `dpi` | Python | 导出像素密度 | PNG 清晰度、文件大小 |
| `fontsize` | Python | 图内文字大小 | 坐标轴、图例、标题 |
| `width` | Quarto | 文档显示宽度 | 最终视觉大小、是否出边 |

最容易被忽略的是：图中文字的最终视觉大小，取决于图形被文档缩放后的结果。近似地说：

$$
\text{visual font size}
\approx
\text{font size}
\times
\frac{\text{display width}}{\text{export width}}
$$

这解释了为什么同样设置 `fontsize=11`，但修改图片宽度和显示宽度后，图例和坐标轴文字会看起来完全不一样。

![](./figs/data_visual_setting_fig03_font_scaling.png){width="92%"}

## 3. 五条绘图排版原则

### 3.1 先确定输出场景，再设置绘图参数

不能只问“这张图多大合适”，而要先问“这张图最后要放在哪里”。

| 场景 | 主要目标 |
|---|---|
| Notebook 预览 | 快速查看，清晰即可 |
| OnlineBook 网页 | 自适应，不出边，不产生横向滚动 |
| A4 PDF | 不超出版心，字号不能太小 |
| 手机端 | 少用复杂横向多图，保证可读性 |
| 课程讲义 | 风格统一，便于长期维护 |

### 3.2 单图、多图必须使用不同尺寸

单图、双图和三图不能共用一套尺寸。单图可以有较充分的横向空间；三图并排时，每个子图的有效宽度会明显变小，需要减少图内文字，并适当降低字号。

![](./figs/data_visual_setting_fig04_single_multi_compare.png){width="96%"}

建议使用以下默认参数：

| 图形类型 | 推荐 `figsize` | 推荐保存 `dpi` | Quarto 显示宽度 |
|---|---:|---:|---:|
| 单图 | `(7.0, 4.2)` | 220 | `85%` |
| 双图并排 | `(9.0, 3.8)` | 220 | `100%` |
| 三图并排 | `(10.5, 3.2)` | 220 | `100%` |
| 2 × 2 多子图 | `(8.5, 6.5)` | 220 | `95%` 或 `100%` |

### 3.3 正文 caption 和图内标题要分工

在 OnlineBook 和论文式文档中，图形说明最好交给正文或 Quarto caption，图内标题只保留必要信息。

推荐做法：

- 单图通常不写图内大标题。
- 多子图可以使用 `(a)`、`(b)`、`(c)` 作为小标题。
- 图内文字越少，跨终端展示越稳。
- 变量名称较长时，优先在正文解释，不要全部塞进坐标轴。

### 3.4 字号、线宽、点大小要一起统一

图形风格统一，不只是统一字号，还要统一线宽、点大小、网格、图例和边框。比如：

```python
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "lines.linewidth": 1.8,
    "lines.markersize": 5,
    "axes.grid": True,
    "grid.alpha": 0.22,
})
```

### 3.5 追求稳健适配，而不是单端最优

一张图很难同时在 PC、iPad、iPhone 和 A4 PDF 中都达到最优效果。实际写作中，更应追求：

- 不出边；
- 不模糊；
- 不拥挤；
- 字号不突兀；
- 多章节风格一致。

## 4. 典型场景参数

### 4.1 单图：正文中的重点展示图

单图适合展示时间序列、分布、拟合关系、单个模型结果等。推荐配置：

```python
figsize = (7.0, 4.2)
save_dpi = 220
```

Quarto 插图建议：

```markdown
![](./figs/data_visual_setting_fig05_single_plot.png){width="85%"}
```

![](./figs/data_visual_setting_fig05_single_plot.png){width="85%"}

### 4.2 双图：适合并排比较

双图适合处理前后对比、两组样本对比、两个模型对比。推荐配置：

```python
figsize = (9.0, 3.8)
save_dpi = 220
```

Quarto 插图建议：

```markdown
![](./figs/data_visual_setting_fig06_two_panel.png){width="100%"}
```

![](./figs/data_visual_setting_fig06_two_panel.png){width="100%"}

### 4.3 三图：适合展示流程，但要谨慎使用

三图适合展示“数据 → 模型 → 结果”的流程，也适合展示三个指标的横向比较。推荐配置：

```python
figsize = (10.5, 3.2)
save_dpi = 220
```

Quarto 插图建议：

```markdown
![](./figs/data_visual_setting_fig07_three_panel.png){width="100%"}
```

![](./figs/data_visual_setting_fig07_three_panel.png){width="100%"}

注意：三图并排在 PC 端比较方便，但在手机端和 A4 PDF 中会被压缩。图内文字要尽量少，图例要短。

### 4.4 2 × 2 多子图：适合模型诊断和多指标比较

2 × 2 多子图适合残差诊断、多指标比较、模型稳健性展示。推荐配置：

```python
figsize = (8.5, 6.5)
save_dpi = 220
```

Quarto 插图建议：

```markdown
![](./figs/data_visual_setting_fig08_four_panel.png){width="95%"}
```

![](./figs/data_visual_setting_fig08_four_panel.png){width="95%"}

## 5. 多章节 OnlineBook 的项目组织

对于多章节 OnlineBook，最常见的问题不是某一张图不好看，而是全书图片风格不统一。建议从一开始就设置统一的绘图配置文件。

![](./figs/data_visual_setting_fig09_project_structure.png){width="92%"}

### 5.1 简单方案：每章保留一份配置代码

适合初学者。每个章节的 Notebook 开头复制相同的全局配置代码。优点是路径简单，不容易出错。

```text
Lecture/
├── chapter_01/
│   ├── lec.ipynb
│   ├── codes.ipynb
│   ├── figs/
│   └── data/
├── chapter_02/
│   ├── lec.ipynb
│   ├── codes.ipynb
│   ├── figs/
│   └── data/
```

### 5.2 进阶方案：全书共用 `plot_config.py`

适合较成熟的项目。把通用配置放到 `_tools/plot_config.py`，各章调用同一套配置。

```text
project/
├── _tools/
│   └── plot_config.py
├── Lecture/
│   ├── chapter_01/
│   └── chapter_02/
```

优点是全书风格更容易统一。缺点是需要处理 Python 的导入路径，对初学者略复杂。

### 5.3 图片命名规则

建议使用：

```text
章节前缀_fig序号_简短说明.png
```

例如：

```text
data_visual_setting_fig01_workflow.png
data_visual_setting_fig02_figsize_dpi_width.png
data_visual_setting_fig03_font_scaling.png
```

这样即使图片未来上传到图床或合并到全书图像目录中，也不容易重名。

## 6. 标准配置代码

下面这段代码适合放在 Notebook 开头。它完成六件事：

- 自动创建 `./figs` 和 `./data`；
- 自动选择中文字体；
- 设置基础绘图风格；
- 提供单图、双图、三图配置函数；
- 同时保存 PNG 和 SVG；
- 固定配色和随机数种子。

```python
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ------------------------------------------------------------
# 1. 创建项目文件夹
# ------------------------------------------------------------

FIG_DIR = Path("./figs")
DATA_DIR = Path("./data")

FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. 中文字体设置
# ------------------------------------------------------------

available_fonts = {f.name for f in fm.fontManager.ttflist}

font_candidates = [
    "SimHei",
    "Microsoft YaHei",
    "Noto Sans CJK SC",
    "Noto Sans CJK JP",
    "WenQuanYi Micro Hei",
    "Arial Unicode MS",
]

FONT_FAMILY = next(
    (font for font in font_candidates if font in available_fonts),
    "DejaVu Sans"
)

plt.rcParams["font.sans-serif"] = [FONT_FAMILY]
plt.rcParams["axes.unicode_minus"] = False

# ------------------------------------------------------------
# 3. 全局基础绘图风格
# ------------------------------------------------------------

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "lines.linewidth": 1.8,
    "lines.markersize": 5,
    "savefig.facecolor": "white",
})

# ------------------------------------------------------------
# 4. 场景切换函数
# ------------------------------------------------------------

def fig_single():
    plt.rcParams.update({
        "figure.figsize": (7.0, 4.2),
        "font.size": 11,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
    })


def fig_double():
    plt.rcParams.update({
        "figure.figsize": (9.0, 3.8),
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 11,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
    })


def fig_triple():
    plt.rcParams.update({
        "figure.figsize": (10.5, 3.2),
        "font.size": 9.5,
        "axes.labelsize": 9.5,
        "axes.titlesize": 10.5,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.fontsize": 8.5,
    })

# ------------------------------------------------------------
# 5. 统一保存函数
# ------------------------------------------------------------

def save_fig(fig, basename, dpi=220):
    fig.savefig(
        FIG_DIR / f"{basename}.png",
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        FIG_DIR / f"{basename}.svg",
        bbox_inches="tight",
        facecolor="white"
    )

# ------------------------------------------------------------
# 6. 配色和随机数
# ------------------------------------------------------------

PALETTE = [
    "#1A3A6B",
    "#C8900A",
    "#2E8B57",
    "#CC2222",
    "#7B3F9E",
    "#666666",
]

RNG = np.random.default_rng(2026)

print(f"当前使用字体: {FONT_FAMILY}")
```

### 6.1 使用方式

画单图前运行：

```python
fig_single()

fig, ax = plt.subplots()
ax.plot(x, y)
save_fig(fig, "chapter_fig01_single")
```

画双图前运行：

```python
fig_double()

fig, axes = plt.subplots(1, 2, constrained_layout=True)
save_fig(fig, "chapter_fig02_double")
```

画三图前运行：

```python
fig_triple()

fig, axes = plt.subplots(1, 3, constrained_layout=True)
save_fig(fig, "chapter_fig03_triple")
```

需要注意：`fig_single()`、`fig_double()` 和 `fig_triple()` 会修改当前 Notebook 后续图形的默认风格。因此，每次改变布局前，最好明确调用一次对应函数。

## 7. 用 AI 生成自己的绘图规范

下面这些提示词适合复制给 AI 或 agent，让它根据你的项目生成配置代码。

::: {.callout-tip}
## 提示词 1：为单个 Notebook 生成绘图配置

我正在使用 Jupyter Notebook 写一篇数据分析文档，后续会通过 Quarto 导出为 HTML 和 A4 PDF。我的图形主要包括单图、双图并排和三图并排。请帮我生成一套 matplotlib 全局配置代码，要求：

- 自动创建 `./figs` 和 `./data` 文件夹；
- 中文字体优先使用 `SimHei`、`Microsoft YaHei` 和 `Noto Sans CJK SC`；
- 单图适合在 Quarto 中设置 `width="85%"`；
- 多图适合设置 `width="100%"`；
- 图中文字要和正文 11 pt 左右的视觉效果协调；
- 代码中加入中文注释。
:::

::: {.callout-tip}
## 提示词 2：为 Quarto OnlineBook 生成全书配置

我正在用 Quarto 搭建多章节 OnlineBook。每章是独立的 `.ipynb` 文件，每章都有大量 matplotlib 图片。请帮我设计一个全书通用的 `plot_config.py`，要求：

- 支持单图、双图、三图和 2 × 2 多子图；
- 统一 `figsize`、`dpi`、字号、线宽、网格和 legend；
- 自动创建每章目录下的 `./figs` 和 `./data`；
- 提供 `save_fig()` 函数，同时保存 PNG 和 SVG；
- 代码适合 Windows 本地和 GitHub Pages 发布流程。
:::

::: {.callout-tip}
## 提示词 3：让 agent 批量修改已有 Notebook

请检查当前文件夹中的所有 `.ipynb` 文件，只修改绘图相关代码，不改变数据处理、模型估计和结果解释。具体要求：

- 统一图片输出到 `./figs`；
- 统一图片命名为 `章节前缀_fig序号_简短说明.png`；
- 单图使用 `figsize=(7.0, 4.2)`；
- 双图使用 `figsize=(9.0, 3.8)`；
- 三图使用 `figsize=(10.5, 3.2)`；
- 保存 PNG 时使用 `dpi=220` 和 `bbox_inches="tight"`；
- 补充必要的中文注释。
:::

::: {.callout-tip}
## 提示词 4：根据输出端生成参数表

我需要同一份 Notebook 同时适配 Quarto OnlineBook、A4 PDF 和手机端阅读。请根据这三个输出场景，给出单图、双图、三图和 2 × 2 多子图的推荐参数表，包括：

- `figsize`；
- `dpi`；
- `font.size`；
- `legend.fontsize`；
- Quarto 中建议使用的 `width`；
- 哪些图形不适合在手机端横向并排展示。
:::

::: {.callout-tip}
## 提示词 5：统一 matplotlib、seaborn 和 plotly 风格

我的项目中混合使用 matplotlib、seaborn 和 plotly。请帮我生成一套统一风格设置，使三类图形在字体、字号、背景、网格、颜色和图例方面尽量一致。要求：

- matplotlib 使用 `plt.rcParams.update()`；
- seaborn 使用 `sns.set_theme()`；
- plotly 使用 `fig.update_layout()`；
- 代码中加入中文注释；
- 保持适合 Quarto OnlineBook 和 A4 PDF 输出。
:::

## 8. 常见问题

### 8.1 图片应该保存为 PNG、SVG 还是 PDF？

| 格式 | 适合场景 | 说明 |
|---|---|---|
| PNG | 网页、推文、复杂热力图 | 兼容性最好，但放大后可能变糊 |
| SVG | 线图、散点图、流程图 | 矢量格式，网页缩放清晰，但中文字体需要检查 |
| PDF | 论文插图、A4 PDF | 适合打印和正式文档 |
| HTML | plotly 交互图 | 适合网页，不适合静态 PDF |

多数教学图可以同时保存 PNG 和 SVG。网页中优先尝试 SVG；如果中文字体或图像元素显示异常，再换 PNG。

### 8.2 为什么 `bbox_inches="tight"` 有时会让图片尺寸不一致？

`bbox_inches="tight"` 会自动裁掉图片周围空白，对单图很方便。但它也可能导致不同图片的外框尺寸不完全一致，尤其是 legend 放在图外时，图片宽度可能被撑大。

建议：

- 单图和普通讲义图可以使用。
- 系列图、并排图和固定版式图要谨慎使用。
- 复杂多子图优先使用 `constrained_layout=True`。

### 8.3 中文字体为什么本地正常，GitHub Pages 或 PDF 中乱码？

原因通常是运行环境不同。Windows 本地有 `SimHei` 或 `Microsoft YaHei`，但 Linux 服务器或 GitHub Actions 环境未必有这些字体。

建议：

- 本地 Windows 可用 `SimHei` 或 `Microsoft YaHei`。
- 跨平台项目优先考虑安装 `Noto Sans CJK SC`。
- SVG 和 PDF 输出后，要实际检查中文是否正常显示。

### 8.4 为什么手机端看三图并排很吃力？

手机屏幕较窄，三图并排会被整体压缩，图中文字也会一起变小。技术推文和 OnlineBook 中，三图并排要谨慎使用。

更稳妥的做法：

- 重要图形使用单图。
- 对比图优先使用双图。
- 三图只用于结构非常清晰、文字很少的场景。
- 手机端阅读优先考虑纵向展示。

### 8.5 Notebook 里好看，保存后为什么变糊？

原因可能有三个：

- Notebook 内联显示有 retina 优化；
- 保存 PNG 时 `dpi` 偏低；
- Quarto 把原图放大显示了。

建议不要只看 Notebook 预览，而要检查最终生成的 PNG、HTML 和 PDF。

## 9. 默认方案

如果不想一开始研究太多细节，可以先使用下面这套参数。

| 图形类型 | Python 参数 | Quarto 参数 |
|---|---|---|
| 单图 | `figsize=(7.0, 4.2), dpi=150, save_dpi=220` | `width="85%"` |
| 双图 | `figsize=(9.0, 3.8), dpi=150, save_dpi=220` | `width="100%"` |
| 三图 | `figsize=(10.5, 3.2), dpi=150, save_dpi=220` | `width="100%"` |
| 2 × 2 图 | `figsize=(8.5, 6.5), dpi=150, save_dpi=220` | `width="95%"` |

本章的核心结论可以概括为三句话：

- Python 负责生成图片，Quarto 负责显示图片。
- 图中文字的最终视觉大小，取决于字号、图形尺寸和文档缩放。
- 多章节项目最好使用统一配置，而不是每个 Notebook 单独设置风格。